In [1]:
import os
from pathlib import Path
import pandas as pd

In [2]:
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [3]:
df = pd.read_csv("data/preprocessed_data.csv")
print(df.shape)
df.head()

(116681, 10)


,review_headline,review_body,star_rating,verified_purchase,helpful_votes,total_votes,problem_category,clean_body,clean_headline,label
0,One Star,Can't finde satelite for DECTVHD Had to get a ...,1,Y,0,0,ürün_dayanıklılığı,can t finde satelite for dectvhd had to get a ...,one star,3
1,One Star,I just got them last week. They don't get full...,1,Y,1,1,ürün_kalitesi,i just got them last week they don t get full ...,one star,4
2,Keep Looking.,This device requires a separate power supply a...,2,Y,0,1,performans,this device requires a separate power supply a...,keep looking,1
3,Barely any channels,Even though this model says it's for 50 miles ...,1,Y,0,0,ürün_dayanıklılığı,even though this model says it s for miles rad...,barely any channels,3
4,did not fit in any actual outlets,This plug did not fit into any outlets I encou...,1,Y,0,0,ürün_kalitesi,this plug did not fit into any outlets i encou...,did not fit in any actual outlets,4


In [4]:
from sklearn.model_selection import train_test_split

X = df["clean_body"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (93344,)
Test: (23337,)


In [6]:
df["clean_body"] = df["clean_body"].fillna("")
X = df["clean_body"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (93344, 10000)
Test shape: (23337, 10000)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)

print(classification_report(y_test, y_pred, target_names=[
    "içerik_beklenti", "performans", "problem_yok", "ürün_dayanıklılığı", "ürün_kalitesi"
]))

                    precision    recall  f1-score   support

   içerik_beklenti       0.90      0.19      0.31       293
        performans       0.83      0.30      0.44       371
       problem_yok       0.95      0.99      0.97     21325
ürün_dayanıklılığı       0.72      0.52      0.60       505
     ürün_kalitesi       0.69      0.42      0.53       843

          accuracy                           0.94     23337
         macro avg       0.82      0.49      0.57     23337
      weighted avg       0.94      0.94      0.93     23337



çerik_beklenti F1: 0.31 — çok kötü

performans F1: 0.44 — kötü

ürün_kalitesi F1: 0.53 — orta


Class imbalance — problem_yok 106k satır, diğerleri 1-4k satır. Model hep problem_yok tahmin ediyor.
    
Çözüm olarak class_weight="balanced" yapacağız

In [9]:
lr_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = lr_balanced.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_balanced, target_names=[
    "içerik_beklenti", "performans", "problem_yok", "ürün_dayanıklılığı", "ürün_kalitesi"
]))

                    precision    recall  f1-score   support

   içerik_beklenti       0.25      0.73      0.37       293
        performans       0.29      0.73      0.42       371
       problem_yok       0.99      0.88      0.93     21325
ürün_dayanıklılığı       0.43      0.78      0.55       505
     ürün_kalitesi       0.38      0.76      0.50       843

          accuracy                           0.87     23337
         macro avg       0.47      0.78      0.56     23337
      weighted avg       0.94      0.87      0.89     23337

